In [ ]:
from transformers import AutoTokenizer
import matplotlib.pyplot as plt
import os
import pandas as pd
from pathlib import Path
from huggingface_hub import login

In [ ]:
def find_project_root(start_path, marker="data"):
    path = Path(start_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

In [ ]:
docs_df = pd.read_csv(DATA_RAW/'transcripts_final.csv')
#docs_df = pd.read_csv('transcripts_testing.csv')


In [ ]:
print(docs_df.columns)

print(docs_df.shape)

In [ ]:
llama_id = "meta-llama/Llama-3.1-8B-Instruct"

llama = AutoTokenizer.from_pretrained(llama_id, )#

def llama_tokens(text):
    return len(llama.encode(text))

In [ ]:
docs_df['llama_tokens'] = docs_df["transcript_text"].apply(llama_tokens)

In [ ]:
docs_df["llama_tokens"].describe()

In [ ]:
docs_df["llama_tokens"].quantile([0.5, 0.75, 0.9, 0.95, .975, .98, .985, 0.99, .995,.999, 1])

In [ ]:
total_tokens = docs_df["llama_tokens"].sum()

print(total_tokens)


In [ ]:
docs_df["mostimportantdateutc"] = pd.to_datetime(
    docs_df["mostimportantdateutc"],
    errors="coerce"   # converts bad values to NaT instead of crashing
)
docs_df["year"] = docs_df["mostimportantdateutc"].dt.year

tokens_per_year = docs_df.groupby("year")["llama_tokens"].sum()
print(tokens_per_year)

In [ ]:
desc= docs_df.groupby("year")["llama_tokens"].agg(
    count="count",
    mean="mean",
    std="std",
    min="min",
    p05 =lambda x: x.quantile(0.05),
    p25=lambda x: x.quantile(0.25),
    median="median",
    p75=lambda x: x.quantile(0.75),
    p95 = lambda x: x.quantile(0.95),
    max="max",
    sum = "sum"
)

print(desc)

desc.to_csv("tokens_summary.csv", sep=";", decimal=",", index=True)

In [ ]:

docs_df["llama_tokens"].hist(bins=40)

plt.xlabel("Tokens per transcript")
plt.ylabel("Frequency")
plt.title("Token distribution of earnings call transcripts")
plt.show()

In [ ]:
docs_df.loc[docs_df['llama_tokens'] > 12000]

In [ ]:
prompt_evaluate = """
You are a financial analyst.

You are given a forward-looking intensity (FLI) score.

Your task is to evaluate the forward-looking economic content in the ENTIRE earnings call Q&A text.

-----------------------
OBJECTIVE
-----------------------

This is a CONDITIONAL evaluation:

1) The Q&A may contain forward-looking information.
2) Evaluate ONLY the forward-looking content in the text.
3) Ignore discussions that are purely historical or not related to future expectations.

Treat the Q&A as a continuous dialogue. Do NOT extract or segment text.

-----------------------
IMPORTANT RULE
-----------------------

Use the provided FLI score as guidance:

- If FLI = 0.00:
  → all numerical values must be 0.00  
  → all categorical fields must be "none"  

- If FLI > 0.00:
  → evaluate the forward-looking content proportionally  
  → low FLI should result in low (but not necessarily zero) scores  
  → high FLI should result in more informative and developed scores  

Do NOT force non-zero values. Scores should reflect the actual strength and usefulness of the forward-looking content.

-----------------------
DIMENSIONS
-----------------------

Evaluate the forward-looking content on:

- specificity: level of detail and precision  
- economic_substance: usefulness for decision-making  
- tone: sentiment (-1 to 1)  
- certainty: confidence and strength of statements  


-----------------------
CATEGORICAL RULES
-----------------------

Select EXACTLY ONE per field. DO NOT create new categories.

main_focus:
strategy OR demand OR costs OR revenue OR supply OR investment OR risk OR product OR market OR competition OR regulation OR operations

secondary_focus:
same list, must differ from main_focus OR "none"

IMPORTANT:
- Assign based ONLY on forward-looking discussion

managerial_horizon:
short_term OR medium_term OR long_term OR mixed OR none

overall_outlook:
negative OR neutral OR positive OR none

DO NOT assign categories based on historical discussion or non-forward-looking content. Focus solely on the forward-looking elements of the Q&A when determining these categorical fields.
-----------------------
OUTPUT FORMAT
-----------------------

Return ONLY valid JSON and DO NOT give explanations. Restrict yourself to the following output format: 

{
  "specificity": 0.00,
  "economic_substance": 0.00,
  "tone": 0.00,
  "certainty": 0.00,
  "context_summary": {
    "main_focus": "",
    "secondary_focus": "",
    "managerial_horizon": "",
    "overall_outlook": ""
  }
}
"""

In [ ]:
prompt_identify = """
You are a financial analyst.

Your task is to evaluate the ENTIRE earnings call Q&A text.

-----------------------
OBJECTIVE
-----------------------

Identify the extent of forward-looking information (FLI) in the FULL text.

Forward-looking information refers to statements about:
- expectations, guidance, or forecasts
- future performance, plans, or strategy
- anticipated risks or opportunities
- outlook on markets, demand, or operations

Do NOT extract or quote specific parts of the text.
Do NOT segment the text.

Evaluate the Q&A as a continuous dialogue.

-----------------------
OUTPUT FORMAT
-----------------------

Return ONLY valid JSON:

{
  "forward_looking_intensity": 0.00
}

-----------------------
SCORING
-----------------------

0.00 = no forward-looking content  
0.25 = very limited forward-looking references  
0.50 = moderate forward-looking discussion  
0.75 = substantial forward-looking content  
1.00 = predominantly forward-looking discussion  

Feel free to score anywhere between these values based on your assessment of the overall forward-looking nature of the Q&A text. Do not round to the nearest quarter if the content does not fit those exact categories. Use your judgment to assign a score that best reflects the forward-looking intensity of the entire Q&A dialogue.

"""

In [ ]:
llama_tokens(prompt_evaluate)

In [ ]:
llama_tokens(prompt_identify)

In [ ]:
docs_df.to_csv(DATA_RAW/"transcripts_final_tokenized.csv", index=False)